# LeetCode 437 — Path Sum III bằng Python

Link đề: https://leetcode.com/problems/path-sum-iii/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree, Hash Table, Prefix Sum

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def pathSum(self, root: Optional[TreeNode], targetSum: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so với cách liệt kê mọi đường đi), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ. Notebook tự dựng `TreeNode` và hàm `build_tree` dựng cây từ list kiểu LeetCode (`[10,5,-3,3,2,null,11,...]`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **đường đi đi xuống** (từ cha tới con, không cần bắt đầu ở gốc hay kết thúc ở lá) có **tổng giá trị bằng `targetSum`**.
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, dựng cây từ list có `None`, đệ quy DFS, `collections.Counter`/`defaultdict`, hàm lồng nhau và `nonlocal`, ngăn xếp tường minh cho DFS không đệ quy.
3. Đi từ **thử mọi điểm bắt đầu rồi đi xuống `O(n²)`** → **giữ đường đi hiện tại và cộng ngược** → **tổng tiền tố + bảng băm (`O(n)`)** → **gộp bảng tổng từ dưới lên** → **bản không đệ quy**.
4. Hiểu mấu chốt: đường đi xuống kết thúc ở nút hiện tại có tổng `= s − (một tổng tiền tố của tổ tiên)`, nên chỉ cần **đếm số lần xuất hiện của mỗi tổng tiền tố** trên đường từ gốc tới nút này — giống bài *Subarray Sum Equals K* nhưng trên cây, với bước **hoàn tác** khi quay lui.
5. Tránh các lỗi kinh điển: quên `cnt[0] = 1`, quên hoàn tác khi quay lui (nhánh này làm bẩn nhánh khác), chèn tổng hiện tại vào bảng **trước** khi tra, và chỉ đếm đường đi bắt đầu từ gốc.

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một cây nhị phân và một số nguyên `targetSum`. Hãy trả về **số đường đi** có tổng các giá trị trên đường đi **bằng `targetSum`**.

Đường đi **không cần** bắt đầu ở gốc hoặc kết thúc ở lá, nhưng phải **đi xuống** (chỉ từ nút cha tới nút con).

**Ví dụ 1**

- Input: `root = [10,5,-3,3,2,null,11,3,-2,null,1]`, `targetSum = 8`
- Output: `3`
- Giải thích: ba đường đi có tổng `8`: `5 -> 3`, `5 -> 2 -> 1`, và `-3 -> 11`.

**Ví dụ 2**

- Input: `root = [5,4,8,11,null,13,4,7,2,null,null,5,1]`, `targetSum = 22`
- Output: `3`
- Giải thích: các đường đi: `5 -> 4 -> 11 -> 2`, `5 -> 8 -> 4 -> 5`, và `4 -> 11 -> 7`.

**Ràng buộc:**

- Số nút của cây nằm trong khoảng `[0, 1000]`.
- `-10^9 <= Node.val <= 10^9`
- `-1000 <= targetSum <= 1000`

Chú ý: giá trị có thể **âm** và có thể bằng `0`, nên không thể dừng sớm khi tổng vượt `targetSum`.

In [1]:
from typing import Optional, List
from collections import Counter, defaultdict, deque
import itertools
import random
import sys


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


ex1 = [10, 5, -3, 3, 2, None, 11, 3, -2, None, 1]
ex2 = [5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1]
examples = [
    (ex1, 8, 3),
    (ex2, 22, 3),
]
for vals, target, expected in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print(vals, "target =", target, "-> kỳ vọng", expected)

[10, 5, -3, 3, 2, None, 11, 3, -2, None, 1] target = 8 -> kỳ vọng 3
[5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1] target = 22 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Nút cây và cây nhị phân

Mỗi nút (`TreeNode`) có `val` (giá trị), `left`, `right` (nút con hoặc `None`). **Chỉ cần giữ nút gốc là truy cập được cả cây.** Ta vừa định nghĩa `TreeNode` ở cell trên (LeetCode cung cấp sẵn, khi nộp bài không cần viết lại).

In [2]:
a = TreeNode(5, TreeNode(4), TreeNode(8))
print(a.val, a.left.val, a.right.val, a.left.left)      # 5 4 8 None

5 4 8 None


### 3.2. Dựng cây từ list kiểu LeetCode

Ví dụ `[10,5,-3,3,2,None,11]` được đọc **theo từng tầng, từ trái sang phải**; `None` nghĩa là "không có nút ở vị trí này" (và **không** có con). Quy tắc dựng: dùng hàng đợi; mỗi nút lấy ra nhận hai phần tử tiếp theo của list làm con trái, con phải. Hàm `build_tree` ở trên làm đúng việc này.

In [3]:
t = build_tree([10, 5, -3, 3, 2, None, 11])
print(t.val, t.left.val, t.right.val)                   # 10 5 -3
print(t.left.left.val, t.left.right.val)                # 3 2
print(t.right.left, t.right.right.val)                  # None 11
print(tree_to_list(t))                                  # [10, 5, -3, 3, 2, None, 11]

10 5 -3
3 2
None 11
[10, 5, -3, 3, 2, None, 11]


### 3.3. DFS đệ quy và thứ tự tiền tự (preorder)

DFS (duyệt theo chiều sâu): làm việc với nút hiện tại, rồi gọi đệ quy cho con trái, con phải. Gọi hàm đệ quy cho `None` thì trả về ngay (**điều kiện dừng**). Mỗi nút ta đi qua **hai lần**: khi đi **xuống** (vào) và khi **quay lên** (ra). Điều này quan trọng: những gì ta ghi lại khi vào, thường phải **hoàn tác khi ra**.

In [4]:
def preorder(node, depth=0):
    if node is None:
        return
    print("  " * depth + f"vào {node.val}")
    preorder(node.left, depth + 1)
    preorder(node.right, depth + 1)
    print("  " * depth + f"ra  {node.val}")


preorder(build_tree([1, 2, 3]))

vào 1
  vào 2
  ra  2
  vào 3
  ra  3
ra  1


### 3.4. Tổng tiền tố (prefix sum) trên một dãy

Với dãy `a`, tổng tiền tố `P[k] = a[0] + ... + a[k-1]` (với `P[0] = 0`). Tổng của đoạn liên tiếp `a[i..j-1]` là `P[j] − P[i]`. Do đó, "có đoạn liên tiếp kết thúc ở `j-1` có tổng `T`" ⇔ "tồn tại `i < j` mà `P[i] = P[j] − T`".

In [5]:
a = [1, 2, 3, -1, 4]
P = [0]
for x in a:
    P.append(P[-1] + x)
print(P)                              # [0, 1, 3, 6, 5, 9]
print(P[4] - P[1], sum(a[1:4]))       # 5 5 : đoạn a[1..3]

[0, 1, 3, 6, 5, 9]
4 4


### 3.5. `Counter`/`dict` để đếm số lần xuất hiện

`Counter` là kiểu `dict` đặc biệt: tra khoá **chưa có** trả về `0` (không báo lỗi), và `cnt[k] += 1` tăng đếm. Đây là công cụ để hỏi "có bao nhiêu tổng tiền tố bằng `x`?" trong `O(1)`.

In [6]:
cnt = Counter({0: 1})
print(cnt[0], cnt[5])           # 1 0   (khoá chưa có -> 0)
cnt[5] += 1
cnt[5] += 1
print(cnt)                      # Counter({5: 2, 0: 1})
cnt[5] -= 1
print(cnt[5])                   # 1

1 0
Counter({5: 2, 0: 1})
1


### 3.6. Hàm lồng nhau và `nonlocal`

Hàm đệ quy `dfs` được định nghĩa **bên trong** `pathSum` thấy được biến của hàm ngoài (như `targetSum`, `cnt`). Muốn **gán lại** một biến số của hàm ngoài (như `total += 1`) thì khai báo `nonlocal total`. Với `dict`/`Counter` chỉ cần sửa nội dung (không gán lại) nên không cần `nonlocal`.

In [7]:
def demo():
    total = 0
    def bump():
        nonlocal total
        total += 1
    bump(); bump()
    return total


print(demo())      # 2

2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét cây nhỏ `[10, 5, -3, 3, 2, None, 11]`, `targetSum = 8`. Liệt kê **mọi đường đi xuống** (mỗi nút là điểm bắt đầu, đi xuống tới mọi nút con cháu) và tổng của nó:

| Điểm bắt đầu | Đường đi | Tổng | `= 8`? |
|---|---|---|---|
| 10 | `10` | 10 | không |
| 10 | `10,5` | 15 | không |
| 10 | `10,-3` | 7 | không |
| 10 | `10,5,3` | 18 | không |
| 10 | `10,5,2` | 17 | không |
| 10 | `10,-3,11` | 18 | không |
| 5 | `5` | 5 | không |
| 5 | `5,3` | 8 | **có** |
| 5 | `5,2` | 7 | không |
| -3 | `-3` | -3 | không |
| -3 | `-3,11` | 8 | **có** |
| 3 / 2 / 11 | `3`, `2`, `11` | 3, 2, 11 | không |

Có `2` đường đi: `5 -> 3` và `-3 -> 11`. Số đường đi xuống của cây `n` nút là cỡ `n × độ sâu`, nên liệt kê hết là chậm với cây sâu.

In [8]:
def all_downward_paths(root):
    """Liệt kê mọi đường đi xuống (list các giá trị). Dùng làm oracle độc lập."""
    out = []
    def go(node, path_to_here):
        if node is None:
            return
        path = path_to_here + [node.val]
        # mọi hậu tố của path (kết thúc tại node) là một đường đi xuống
        for k in range(len(path)):
            out.append(path[k:])
        go(node.left, path)
        go(node.right, path)
    go(root, [])
    return out


t = build_tree([10, 5, -3, 3, 2, None, 11])
paths = all_downward_paths(t)
print(len(paths), "đường đi")
print([p for p in paths if sum(p) == 8])

14 đường đi
[[5, 3], [-3, 11]]


## 5. Cách 1 — Brute Force / baseline: mỗi nút làm điểm bắt đầu

**Ý tưởng.** Hai tầng đệ quy:

1. `count_from(node, remain)`: đếm số đường đi **bắt đầu đúng ở `node`** và đi xuống có tổng `remain`. Ở mỗi nút trừ `node.val` khỏi `remain`; nếu về `0` thì tính một đường đi (vẫn **tiếp tục đi xuống** vì giá trị âm/`0` có thể tạo thêm đường đi dài hơn).
2. `pathSum(root)`: tổng của `count_from(root, target)` + đệ quy cho con trái và con phải (làm điểm bắt đầu ở mọi nút).

In [9]:
def path_sum_bruteforce(root, target):
    def count_from(node, remain):
        if node is None:
            return 0
        remain -= node.val
        return (remain == 0) + count_from(node.left, remain) + count_from(node.right, remain)

    if root is None:
        return 0
    return (count_from(root, target)
            + path_sum_bruteforce(root.left, target)
            + path_sum_bruteforce(root.right, target))


for vals, target, expected in examples:
    print(path_sum_bruteforce(build_tree(vals), target), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Mỗi nút làm điểm bắt đầu và duyệt cả cây con của nó: tổng là tổng kích thước mọi cây con. Cây cân bằng: **`O(n log n)`**; cây suy biến thành đường thẳng (độ sâu `n`): **Time `O(n²)`**. **Space `O(h)`** cho ngăn xếp đệ quy (`h` là chiều cao).

**Nút thắt:** từ nút `u` ta đi xuống, rồi từ con của `u` lại đi xuống tiếp phần đã đi — **lặp lại công việc**. Phải có cách "nhớ" các tổng đã tích luỹ trên đường từ gốc xuống.

## 6. Ý tưởng thuật toán / bước đột phá

**Đường đi xuống kết thúc tại nút `v` = một đoạn liên tiếp trên đường từ gốc tới `v`.** Gọi `S(v)` là tổng các giá trị từ gốc tới `v` (tổng tiền tố trên đường đó) và `S(u)` là của tổ tiên `u` (hoặc `0` nếu đường đi bắt đầu tại gốc). Đường đi từ **con của `u`** xuống `v` có tổng `S(v) − S(u)`. Muốn bằng `targetSum` thì cần `S(u) = S(v) − targetSum`.

Vậy: khi đứng ở `v`, số đường đi **kết thúc tại `v`** = **số tổ tiên `u` (kể cả "gốc ảo" có `S = 0`) mà `S(u) = S(v) − targetSum`**. Nếu ta giữ một bảng đếm `cnt[x]` = số tổng tiền tố bằng `x` **trên đường từ gốc tới cha của `v`**, thì đáp án tại `v` là `cnt[S(v) − targetSum]`.

Ba chi tiết quyết định:

1. **Khởi tạo `cnt[0] = 1`:** đại diện "tổng tiền tố trước khi đi vào gốc", để đường đi **bắt đầu ngay ở gốc** được đếm.
2. **Tra trước, thêm sau:** tra `cnt[S − target]` **trước** khi thêm `S` vào bảng; nếu không, khi `target = 0` ta sẽ ghép `v` với chính nó (đường đi rỗng).
3. **Hoàn tác khi quay lên:** bảng chỉ được phản ánh **đường từ gốc tới nút hiện tại**; sau khi xử lý xong cây con của `v`, phải `cnt[S] -= 1` để nhánh anh em không "thấy" các tổng của nhánh này.

In [10]:
# Mô phỏng thủ công trên đường 10 -> 5 -> 3, target = 8
target = 8
cnt = Counter({0: 1})
s = 0
for v in [10, 5, 3]:
    s += v
    ways = cnt[s - target]
    print(f"nút {v}: S={s}, cần S-u = {s - target}, cnt[{s - target}] = {ways}")
    cnt[s] += 1
print("cnt =", dict(cnt))

nút 10: S=10, cần S-u = 2, cnt[2] = 0
nút 5: S=15, cần S-u = 7, cnt[7] = 0
nút 3: S=18, cần S-u = 10, cnt[10] = 1
cnt = {0: 1, 10: 1, 15: 1, 18: 1}


## 7. Các cách tiếp theo

### Cách 2 — Giữ đường đi hiện tại, cộng ngược các hậu tố (`O(n · h)`)

**Giải thích từng đoạn.**

1. Giữ một list `path` các giá trị từ gốc tới nút hiện tại (`append` khi vào, `pop` khi ra).
2. Tại mỗi nút, sau khi `append(node.val)`: duyệt `path` **từ cuối về đầu**, cộng dồn `running`; mỗi lần `running == target` thì tính một đường đi (hậu tố của `path` là một đường đi kết thúc tại nút hiện tại).
3. Đệ quy hai con rồi `pop`.

Khác Cách 1: không đi xuống lặp lại nhiều lần từ từng điểm bắt đầu; ta đi **một lượt** và mỗi nút chỉ nhìn **ngược lên** các tổ tiên (tối đa `h`).

In [11]:
def path_sum_suffix(root, target):
    path = []
    total = 0

    def dfs(node):
        nonlocal total
        if node is None:
            return
        path.append(node.val)
        running = 0
        for k in range(len(path) - 1, -1, -1):       # cộng ngược từ nút hiện tại lên tổ tiên
            running += path[k]
            if running == target:
                total += 1
        dfs(node.left)
        dfs(node.right)
        path.pop()

    dfs(root)
    return total


for vals, target, expected in examples:
    print(path_sum_suffix(build_tree(vals), target), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
3 (kỳ vọng 3 )


In [12]:
# Code nộp LeetCode cho Cách 2 (cây suy biến sâu 1000 -> O(n^2) ~ 5*10^5 phép cộng, vẫn chạy được)
class SolutionSuffix:
    def pathSum(self, root: Optional[TreeNode], targetSum: int) -> int:
        path = []
        total = 0

        def dfs(node):
            nonlocal total
            if node is None:
                return
            path.append(node.val)
            running = 0
            for k in range(len(path) - 1, -1, -1):
                running += path[k]
                if running == targetSum:
                    total += 1
            dfs(node.left)
            dfs(node.right)
            path.pop()

        dfs(root)
        return total


print(SolutionSuffix().pathSum(build_tree(ex1), 8))      # 3

3


**Độ phức tạp Cách 2:** tại nút ở độ sâu `d` ta duyệt `d` phần tử ⇒ tổng `Σ d` = **Time `O(n · h)`** (cây cân bằng `O(n log n)`, cây suy biến `O(n²)`), **Space `O(h)`**.

### Cách 3 — Tổng tiền tố + bảng băm (`O(n)`) — cách chuẩn

**Giải thích từng đoạn.** (xem mục 6)

1. `cnt = Counter({0: 1})`.
2. `dfs(node, s)` trả về số đường đi kết thúc ở `node` hoặc trong cây con của `node`:
   - `node is None` ⇒ `0`.
   - `s += node.val`; `ans = cnt[s − targetSum]`.
   - `cnt[s] += 1`; `ans += dfs(left, s) + dfs(right, s)`; `cnt[s] -= 1` (hoàn tác).
3. Trả `dfs(root, 0)`.

In [13]:
def path_sum_prefix(root, target):
    cnt = Counter({0: 1})

    def dfs(node, s):
        if node is None:
            return 0
        s += node.val
        ans = cnt[s - target]            # đường đi kết thúc tại node
        cnt[s] += 1
        ans += dfs(node.left, s)
        ans += dfs(node.right, s)
        cnt[s] -= 1                      # hoàn tác khi quay lên
        return ans

    return dfs(root, 0)


for vals, target, expected in examples:
    print(path_sum_prefix(build_tree(vals), target), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
3 (kỳ vọng 3 )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionPrefix:
    def pathSum(self, root: Optional[TreeNode], targetSum: int) -> int:
        def dfs(node, s):
            if node is None:
                return 0
            s += node.val
            ans = cnt[s - targetSum]
            cnt[s] += 1
            ans += dfs(node.left, s)
            ans += dfs(node.right, s)
            cnt[s] -= 1
            return ans

        cnt = Counter({0: 1})
        return dfs(root, 0)


print(SolutionPrefix().pathSum(build_tree(ex2), 22))      # 3

3


**Độ phức tạp Cách 3:** mỗi nút được thăm **một lần**, các thao tác trên `Counter` là `O(1)` trung bình ⇒ **Time `O(n)`**, **Space `O(n)`** (bảng đếm tối đa `h` khoá khác nhau nhưng `h` có thể `n`; cộng ngăn xếp đệ quy `O(h)`).

### Cách 4 — Gộp bảng tổng từ dưới lên (bottom-up)

**Giải thích từng đoạn.**

1. `dfs(node)` trả về một `Counter` `down`: `down[x]` = số đường đi xuống **bắt đầu tại `node`** có tổng `x`.
2. Với nút `node`: lấy `left = dfs(node.left)`, `right = dfs(node.right)`. Đường đi bắt đầu tại `node` là: chỉ `node` (tổng `node.val`), hoặc `node` nối với một đường đi bắt đầu tại con (tổng `node.val + x`).
3. `down = {node.val: 1} ∪ {node.val + x: c for (x, c) in left} ∪ {... right}`. Mỗi lần dựng xong `down`, cộng `down[target]` vào đáp án chung.

Đây là cách nghĩ **ngược** với Cách 3: không nhìn lên tổ tiên mà để mỗi cây con "báo cáo" lên cha tất cả tổng nó có thể tạo ra.

In [15]:
def path_sum_bottom_up(root, target):
    total = 0

    def dfs(node):
        nonlocal total
        if node is None:
            return {}
        down = defaultdict(int)
        down[node.val] += 1                                  # đường đi chỉ gồm node
        for child in (node.left, node.right):
            for x, c in dfs(child).items():
                down[node.val + x] += c                      # node nối thêm đường đi của con
        total += down.get(target, 0)
        return down

    dfs(root)
    return total


for vals, target, expected in examples:
    print(path_sum_bottom_up(build_tree(vals), target), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
3 (kỳ vọng 3 )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionBottomUp:
    def pathSum(self, root: Optional[TreeNode], targetSum: int) -> int:
        total = 0

        def dfs(node):
            nonlocal total
            if node is None:
                return {}
            down = defaultdict(int)
            down[node.val] += 1
            for child in (node.left, node.right):
                for x, c in dfs(child).items():
                    down[node.val + x] += c
            total += down.get(targetSum, 0)
            return down

        dfs(root)
        return total


print(SolutionBottomUp().pathSum(build_tree(ex1), 8))      # 3

3


**Độ phức tạp Cách 4:** bảng của một nút có tối đa `(số nút trong cây con)` khoá, gộp mất kích thước đó ⇒ trong trường hợp xấu **Time `O(n · h)`** (hay `O(n²)` với cây suy biến), **Space `O(n)`** cho các bảng đang sống. Chậm hơn Cách 3, nhưng cho thấy một góc nhìn khác.

### Cách 5 — Tổng tiền tố + bảng băm, DFS **không đệ quy**

**Giải thích từng đoạn.**

1. Cây sâu `1000` nút có thể chạm giới hạn đệ quy mặc định của Python (`~1000`), nên ta dùng **ngăn xếp tường minh**.
2. Mỗi phần tử trên ngăn xếp là `(node, s_cha, trạng_thái)`: trạng thái `0` = lần đầu thăm (vào), `1` = quay lên (ra).
3. Khi **vào**: `s = s_cha + node.val`, cộng `cnt[s − target]` vào đáp án, `cnt[s] += 1`, đẩy lại phần tử "ra" rồi đẩy hai con ("vào").
4. Khi **ra**: `cnt[s] -= 1`.

In [17]:
def path_sum_iterative(root, target):
    if root is None:
        return 0
    cnt = Counter({0: 1})
    total = 0
    stack = [(root, 0, 0)]                     # (node, tổng tiền tố của cha, 0=vào / 1=ra)
    while stack:
        node, s_parent, state = stack.pop()
        s = s_parent + node.val
        if state == 0:
            total += cnt[s - target]
            cnt[s] += 1
            stack.append((node, s_parent, 1))   # sẽ quay lại để hoàn tác
            if node.right:
                stack.append((node.right, s, 0))
            if node.left:
                stack.append((node.left, s, 0))
        else:
            cnt[s] -= 1
    return total


for vals, target, expected in examples:
    print(path_sum_iterative(build_tree(vals), target), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
3 (kỳ vọng 3 )


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionIterative:
    def pathSum(self, root: Optional[TreeNode], targetSum: int) -> int:
        if root is None:
            return 0
        cnt = Counter({0: 1})
        total = 0
        stack = [(root, 0, 0)]
        while stack:
            node, s_parent, state = stack.pop()
            s = s_parent + node.val
            if state == 0:
                total += cnt[s - targetSum]
                cnt[s] += 1
                stack.append((node, s_parent, 1))
                if node.right:
                    stack.append((node.right, s, 0))
                if node.left:
                    stack.append((node.left, s, 0))
            else:
                cnt[s] -= 1
        return total


print(SolutionIterative().pathSum(build_tree(ex2), 22))      # 3

3


**Độ phức tạp Cách 5:** **Time `O(n)`** (mỗi nút vào/ra đúng một lần), **Space `O(n)`** (ngăn xếp và `Counter`). Không bị `RecursionError` với cây rất sâu.

## 8. TRACE từng bước

Ta trace Cách 3 (tổng tiền tố + bảng băm) trên cây nhỏ `[1, 2, 3, -1]` để thấy rõ cả bước **vào**, **tra bảng**, **hoàn tác**. Cây: `1` có con trái `2` và con phải `3`; `2` có con trái `-1`. `targetSum = 3`.

In [19]:
def trace_prefix(root, target):
    cnt = Counter({0: 1})
    total = [0]

    def dfs(node, s, depth):
        if node is None:
            return
        pad = "  " * depth
        s += node.val
        hit = cnt[s - target]
        total[0] += hit
        print(f"{pad}vào {node.val:>2}: S={s:>2}, cần cnt[{s - target:>2}] = {hit}, tổng đáp án = {total[0]}")
        cnt[s] += 1
        print(f"{pad}      thêm S={s}: cnt = { {k: v for k, v in cnt.items() if v} }")
        dfs(node.left, s, depth + 1)
        dfs(node.right, s, depth + 1)
        cnt[s] -= 1
        print(f"{pad}ra   {node.val:>2}: hoàn tác S={s}: cnt = { {k: v for k, v in cnt.items() if v} }")

    dfs(root, 0, 0)
    return total[0]


assert trace_prefix(build_tree([1, 2, 3, -1]), 3) == 2

vào  1: S= 1, cần cnt[-2] = 0, tổng đáp án = 0
      thêm S=1: cnt = {0: 1, 1: 1}
  vào  2: S= 3, cần cnt[ 0] = 1, tổng đáp án = 1
        thêm S=3: cnt = {0: 1, 1: 1, 3: 1}
    vào -1: S= 2, cần cnt[-1] = 0, tổng đáp án = 1
          thêm S=2: cnt = {0: 1, 1: 1, 3: 1, 2: 1}
    ra   -1: hoàn tác S=2: cnt = {0: 1, 1: 1, 3: 1}
  ra    2: hoàn tác S=3: cnt = {0: 1, 1: 1}
  vào  3: S= 4, cần cnt[ 1] = 1, tổng đáp án = 2
        thêm S=4: cnt = {0: 1, 1: 1, 4: 1}
  ra    3: hoàn tác S=4: cnt = {0: 1, 1: 1}
ra    1: hoàn tác S=1: cnt = {0: 1}


In [20]:
def trace_suffix(root, target):
    path, total = [], [0]

    def dfs(node, depth):
        if node is None:
            return
        path.append(node.val)
        running, found = 0, []
        for k in range(len(path) - 1, -1, -1):
            running += path[k]
            if running == target:
                found.append(path[k:])
        total[0] += len(found)
        print("  " * depth + f"path={path}, đường đi hợp lệ kết thúc tại {node.val}: {found}")
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)
        path.pop()

    dfs(root, 0)
    return total[0]


assert trace_suffix(build_tree([1, 2, 3, -1]), 3) == 2

path=[1], đường đi hợp lệ kết thúc tại 1: []
  path=[1, 2], đường đi hợp lệ kết thúc tại 2: [[1, 2]]
    path=[1, 2, -1], đường đi hợp lệ kết thúc tại -1: []
  path=[1, 3], đường đi hợp lệ kết thúc tại 3: [[3]]


Bảng trace Cách 3 trên cây `[1,2,3,-1]`, `targetSum = 3` (đi theo `1 -> 2 -> -1`, rồi sang nhánh `1 -> 3`):

| Bước | Nút | `S` | `S − target` | `cnt[S − target]` | Ghi chú | Đáp án |
|---|---|---|---|---|---|---|
| vào | 1 | 1 | -2 | 0 | sau đó `cnt = {0:1, 1:1}` | 0 |
| vào | 2 | 3 | 0 | 1 | đường `1 -> 2` (hoặc tính từ gốc ảo) | 1 |
| vào | -1 | 2 | -1 | 0 | `cnt = {0:1,1:1,3:1,2:1}` | 1 |
| ra | -1 | | | | hoàn tác `cnt[2]` | 1 |
| ra | 2 | | | | hoàn tác `cnt[3]` | 1 |
| vào | 3 | 4 | 1 | 1 | tổ tiên `1` có `S=1`: đường `3` đơn lẻ | 2 |
| ra | 3 | | | | hoàn tác `cnt[4]` | 2 |

Kết quả `2`: đường `1 -> 2` (tổng `3`, khớp với gốc ảo `S = 0`) và đường đơn `[3]` (tổng `3`, khớp với tổ tiên `1` có `S = 1`). Bạn có thể đối chiếu với danh sách `found` mà `trace_suffix` in ra ở cell trước.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (tổng tiền tố + `Counter`, đệ quy): `O(n)`, ngắn gọn.

In [21]:
class Solution:
    def pathSum(self, root: Optional[TreeNode], targetSum: int) -> int:
        def dfs(node, s):
            if node is None:
                return 0
            s += node.val                      # tổng tiền tố từ gốc tới node
            ans = cnt[s - targetSum]           # số đường đi kết thúc tại node
            cnt[s] += 1                        # ghi nhận s cho các hậu duệ
            ans += dfs(node.left, s)
            ans += dfs(node.right, s)
            cnt[s] -= 1                        # hoàn tác khi quay lên
            return ans

        cnt = Counter({0: 1})                  # tổng tiền tố "trước gốc" = 0 xuất hiện 1 lần
        return dfs(root, 0)


sol = Solution()
for vals, target, expected in examples:
    print(vals, target, "->", sol.pathSum(build_tree(vals), target))

[10, 5, -3, 3, 2, None, 11, 3, -2, None, 1] 8 -> 3
[5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1] 22 -> 3


## 10. Vì sao thuật toán đúng

- **Một-một giữa đường đi và cặp tổ tiên:** mỗi đường đi xuống được xác định duy nhất bởi nút **cuối** `v` và nút **trước điểm bắt đầu** `u` (một tổ tiên của `v`, hoặc "gốc ảo"). Đếm đường đi kết thúc tại `v` chính là đếm các `u` hợp lệ trên đường từ gốc tới `v`.
- **Công thức:** tổng đường đi là `S(v) − S(u)`; bằng `target` khi `S(u) = S(v) − target`. `cnt[x]` là số tổ tiên có `S = x`, nên đáp án tại `v` là `cnt[S(v) − target]`.
- **`cnt[0] = 1`:** cho phép `u` là "gốc ảo", tức đường đi bắt đầu **ngay tại gốc**.
- **Tra trước, thêm sau:** bảng chỉ chứa **tổ tiên thực sự** (không chứa `v`), nên không đếm đường đi rỗng khi `target = 0`.
- **Hoàn tác:** khi quay lên khỏi `v`, `S(v)` không còn là tổng của một tổ tiên đối với các nút khác nhánh; trừ đi `1` khôi phục đúng tập tổ tiên.
- **Giá trị âm / `0`:** không dùng bất kỳ giả thiết đơn điệu nào (không "cắt tỉa" khi tổng vượt `target`), nên đúng với mọi dấu.
- **Đối chiếu độc lập:** mục 12 so sánh với oracle "liệt kê mọi đường đi rồi đếm" trên hàng nghìn cây ngẫu nhiên.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là chiều cao cây.

| Cách | Time | Space |
|---|---|---|
| 1. Mỗi nút làm điểm bắt đầu | `O(n · h)` (`O(n log n)` cân bằng, `O(n²)` suy biến) | `O(h)` |
| 2. Giữ đường đi + cộng ngược | `O(n · h)` | `O(h)` |
| 3. Tổng tiền tố + hash (đệ quy) | `O(n)` | `O(n)` |
| 4. Gộp bảng từ dưới lên | `O(n · h)` | `O(n)` |
| 5. Tổng tiền tố + hash (lặp) | `O(n)` | `O(n)` |

Với `n <= 1000`, mọi cách đều nhanh, nhưng Cách 3 và 5 là tối ưu về lý thuyết (tuyến tính).

## 12. Bộ test

Ta gom các cách vào `impls`; mỗi hàm nhận **cây** và `target`. Oracle độc lập: `oracle(root, target)` đếm bằng cách **liệt kê mọi đường đi xuống** (`all_downward_paths`).

In [22]:
def oracle(root, target):
    return sum(1 for p in all_downward_paths(root) if sum(p) == target)


impls = {
    "bruteforce": path_sum_bruteforce,
    "suffix": path_sum_suffix,
    "prefix": path_sum_prefix,
    "bottom_up": path_sum_bottom_up,
    "iterative": path_sum_iterative,
    "leetcode": lambda r, t: Solution().pathSum(r, t),
    "submit_suffix": lambda r, t: SolutionSuffix().pathSum(r, t),
    "submit_bottom_up": lambda r, t: SolutionBottomUp().pathSum(r, t),
    "submit_iter": lambda r, t: SolutionIterative().pathSum(r, t),
}


def check_all(values, target, expected, label):
    root = build_tree(values)
    assert oracle(root, target) == expected, (label, "oracle")
    for key, fn in impls.items():
        actual = fn(root, target)
        assert actual == expected, (label, key, values, target, actual, expected)
    print("PASS", label)


for idx, (vals, target, expected) in enumerate(examples, 1):
    check_all(vals, target, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

In [23]:
check_all([], 0, 0, "cây rỗng")
check_all([1], 1, 1, "một nút bằng target")
check_all([1], 2, 0, "một nút khác target")
check_all([0], 0, 1, "một nút 0, target 0")
check_all([0, 0, 0], 0, 5, "ba nút 0: mọi đường đi đều có tổng 0 (3 đơn + 2 cặp gốc-con)")
check_all([1, -1], 0, 1, "đường đi 1 -> -1")
check_all([1, -1, None, 1, None, -1], 0, 4, "đường chéo +1 -1 +1 -1")
check_all([1, 2, None, 3, None, 4], 6, 1, "chuỗi trái 1-2-3-4: chỉ có 1+2+3 = 6 (2+3+4 = 9, 1+2+3+4 = 10)")
check_all([10**9, 10**9, None, -10**9], 10**9, 3, "giá trị lớn, tổng vượt 32-bit")
check_all([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1], 0, 0, "target 0 không đường nào")
check_all([-1, -2, -3], -3, 2, "toàn số âm: -3 và -1,-2")

PASS cây rỗng
PASS một nút bằng target
PASS một nút khác target
PASS một nút 0, target 0
PASS ba nút 0: mọi đường đi đều có tổng 0 (3 đơn + 2 cặp gốc-con)
PASS đường đi 1 -> -1
PASS đường chéo +1 -1 +1 -1
PASS chuỗi trái 1-2-3-4: chỉ có 1+2+3 = 6 (2+3+4 = 9, 1+2+3+4 = 10)
PASS giá trị lớn, tổng vượt 32-bit
PASS target 0 không đường nào
PASS toàn số âm: -3 và -1,-2


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) chỉ đếm đường đi **bắt đầu từ gốc**; (b) **quên hoàn tác** `cnt[s] -= 1`. Ta tạo cây khiến cả hai cho kết quả sai.

In [24]:
def wrong_only_from_root(root, target):
    def count_from(node, remain):
        if node is None:
            return 0
        remain -= node.val
        return (remain == 0) + count_from(node.left, remain) + count_from(node.right, remain)
    return count_from(root, target)              # SAI: không thử các nút khác làm điểm bắt đầu


def wrong_no_undo(root, target):
    cnt = Counter({0: 1})

    def dfs(node, s):
        if node is None:
            return 0
        s += node.val
        ans = cnt[s - target]
        cnt[s] += 1
        ans += dfs(node.left, s)
        ans += dfs(node.right, s)
        return ans                               # SAI: thiếu cnt[s] -= 1

    return dfs(root, 0)


tree = build_tree(ex1)
print("chỉ từ gốc:", wrong_only_from_root(tree, 8), "| đúng:", Solution().pathSum(tree, 8))
assert wrong_only_from_root(tree, 8) != 3

# nhánh trái và nhánh phải có cùng tổng tiền tố nhưng là anh em: quên hoàn tác làm nhánh phải "thấy" nhánh trái
tree2 = build_tree([0, 1, 1])
print("không hoàn tác:", wrong_no_undo(tree2, 0), "| đúng:", Solution().pathSum(tree2, 0))
assert wrong_no_undo(tree2, 0) != Solution().pathSum(tree2, 0)

chỉ từ gốc: 0 | đúng: 3
không hoàn tác: 2 | đúng: 1


### 12.3. Đối chiếu ngẫu nhiên (có seed) và đối chiếu với oracle

Sinh cây ngẫu nhiên đúng ràng buộc: số nút `0..1000`, giá trị trong `[-10^9, 10^9]` (với cây nhỏ dùng giá trị bé để dễ có đường đi khớp), `targetSum` trong `[-1000, 1000]`. Cây được tạo bằng cách gắn dần nút mới vào một chỗ trống ngẫu nhiên; ta thử cả dạng "cao gầy" (chuỗi) lẫn "thấp bè". Với mỗi cây ta chọn `target` là **tổng của một đường đi có thật** để chắc chắn có kết quả dương.

In [25]:
rng = random.Random(437)


def random_tree(n, lo, hi, chain_bias=0.0):
    """Cây n nút; chain_bias càng lớn cây càng giống đường thẳng."""
    if n == 0:
        return None
    root = TreeNode(rng.randint(lo, hi))
    nodes = [root]
    slots = [(root, "left"), (root, "right")]
    last = None
    for _ in range(n - 1):
        if last is not None and rng.random() < chain_bias:
            pool = [s for s in slots if s[0] is last] or slots
        else:
            pool = slots
        parent, side = rng.choice(pool)
        slots.remove((parent, side))
        node = TreeNode(rng.randint(lo, hi))
        setattr(parent, side, node)
        slots += [(node, "left"), (node, "right")]
        last = node
    return root


for trial in range(2500):
    n = rng.randint(0, 14)
    root = random_tree(n, -4, 4, chain_bias=rng.choice([0, 0.5, 0.9]))
    paths = all_downward_paths(root)
    if paths and rng.random() < 0.7:
        target = sum(rng.choice(paths))
    else:
        target = rng.randint(-10, 10)
    expected = oracle(root, target)
    for key, fn in impls.items():
        assert fn(root, target) == expected, (key, tree_to_list(root), target)
print("PASS 2500 cây nhỏ so với oracle liệt kê đường đi")

sys.setrecursionlimit(10000)
for trial in range(40):
    n = rng.randint(100, 600)
    root = random_tree(n, -3, 3, chain_bias=rng.choice([0, 0.6, 0.95]))
    target = rng.randint(-8, 8)
    ref = oracle(root, target)
    for key, fn in impls.items():
        assert fn(root, target) == ref, (key, n)
print("PASS 40 cây vừa (100..600 nút)")

for trial in range(10):
    root = random_tree(1000, -10**9, 10**9, chain_bias=0.5)
    target = rng.randint(-1000, 1000)
    ref = path_sum_iterative(root, target)
    assert Solution().pathSum(root, target) == ref == path_sum_suffix(root, target)
print("PASS 10 cây 1000 nút giá trị lớn")

PASS 2500 cây nhỏ so với oracle liệt kê đường đi


PASS 40 cây vừa (100..600 nút)
PASS 10 cây 1000 nút giá trị lớn


In [26]:
# Cây suy biến sâu 1000 (chuỗi): kiểm tra hiệu năng và giới hạn đệ quy
depth = 1000
chain = TreeNode(1)
cur = chain
for _ in range(depth - 1):
    cur.left = TreeNode(1)
    cur = cur.left

print("đệ quy (limit 10000):", Solution().pathSum(chain, 10))          # mọi đoạn dài 10: 991
print("lặp:", SolutionIterative().pathSum(chain, 10))
assert Solution().pathSum(chain, 10) == SolutionIterative().pathSum(chain, 10) == depth - 10 + 1
assert SolutionSuffix().pathSum(chain, 10) == depth - 10 + 1

đệ quy (limit 10000): 991
lặp: 991


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ đếm các đường đi bắt đầu từ gốc

In [27]:
def sai_chi_tu_goc(root, target):
    if root is None:
        return 0
    def go(node, remain):
        if node is None:
            return 0
        remain -= node.val
        return (remain == 0) + go(node.left, remain) + go(node.right, remain)
    return go(root, target)


tree = build_tree(ex1)
print("sai:", sai_chi_tu_goc(tree, 8), "| đúng:", Solution().pathSum(tree, 8))

sai: 0 | đúng: 3


**Vì sao sai:** đề nói đường đi **không cần** bắt đầu ở gốc (`5 -> 3` bắt đầu ở nút `5`). **Cách sửa:** thử mọi nút làm điểm bắt đầu (Cách 1), hoặc dùng tổng tiền tố (Cách 3).

### Lỗi 2: Quên `cnt[0] = 1`

In [28]:
def sai_quen_cnt0(root, target):
    cnt = Counter()                                    # SAI: thiếu cnt[0] = 1

    def dfs(node, s):
        if node is None:
            return 0
        s += node.val
        ans = cnt[s - target]
        cnt[s] += 1
        ans += dfs(node.left, s) + dfs(node.right, s)
        cnt[s] -= 1
        return ans

    return dfs(root, 0)


tree = build_tree([5])
print("sai:", sai_quen_cnt0(tree, 5), "| đúng:", Solution().pathSum(tree, 5))
tree = build_tree([1, 2])
print("sai:", sai_quen_cnt0(tree, 3), "| đúng:", Solution().pathSum(tree, 3))     # đường 1 -> 2 bắt đầu ở gốc

sai: 0 | đúng: 1
sai: 0 | đúng: 1


**Vì sao sai:** các đường đi bắt đầu **ngay tại gốc** ứng với `S(v) − target = 0`; phải có `cnt[0] = 1` làm "tổng tiền tố trước gốc". **Cách sửa:** `Counter({0: 1})`.

### Lỗi 3: Thêm tổng hiện tại vào bảng trước khi tra

In [29]:
def sai_them_truoc_khi_tra(root, target):
    cnt = Counter({0: 1})

    def dfs(node, s):
        if node is None:
            return 0
        s += node.val
        cnt[s] += 1                                   # SAI: thêm trước
        ans = cnt[s - target]
        ans += dfs(node.left, s) + dfs(node.right, s)
        cnt[s] -= 1
        return ans

    return dfs(root, 0)


tree = build_tree([3, 1, 2])
print("target = 0, sai:", sai_them_truoc_khi_tra(tree, 0), "| đúng:", Solution().pathSum(tree, 0))

target = 0, sai: 3 | đúng: 0


**Vì sao sai:** khi `target = 0`, `s − target = s`; nếu đã thêm `s` thì nút ghép với **chính nó** và đếm một "đường đi rỗng" (tổng 0) cho mỗi nút. **Cách sửa:** tra `cnt[s - target]` **trước**, rồi mới `cnt[s] += 1`.

### Lỗi 4: Quên hoàn tác khi quay lên

In [30]:
def sai_quen_hoan_tac(root, target):
    cnt = Counter({0: 1})

    def dfs(node, s):
        if node is None:
            return 0
        s += node.val
        ans = cnt[s - target]
        cnt[s] += 1
        ans += dfs(node.left, s) + dfs(node.right, s)
        return ans                                    # SAI: không cnt[s] -= 1

    return dfs(root, 0)


tree = build_tree([0, 1, 1])
print("sai:", sai_quen_hoan_tac(tree, 0), "| đúng:", Solution().pathSum(tree, 0))

sai: 2 | đúng: 1


**Vì sao sai:** hai nút `1` ở hai nhánh anh em cùng có `S = 1`. Sau khi xử lý nhánh trái mà không trừ lại, nhánh phải **thấy nhầm** tổng tiền tố của nhánh trái và đếm đường đi không tồn tại (nút phải không có tổ tiên trên đường của nhánh trái). **Cách sửa:** `cnt[s] -= 1` sau khi xử lý hai con (đúng nguyên tắc quay lui).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Đếm đoạn có tổng bằng `k`**" trên dãy ⇒ tổng tiền tố + bảng đếm (Subarray Sum Equals K). Trên **cây**, "dãy" là **đường từ gốc tới nút hiện tại**, nên thêm bước **hoàn tác** khi quay lên.
- Khung mẫu: `cnt = Counter({0: 1})`; vào nút: `s += val; ans += cnt[s - k]; cnt[s] += 1`; ra nút: `cnt[s] -= 1`.
- Có số **âm** hoặc `0` ⇒ không thể dùng cửa sổ trượt hai con trỏ, phải dùng tổng tiền tố.
- Cây sâu (cỡ `10^5` nút suy biến) ⇒ dùng ngăn xếp tường minh hoặc tăng `sys.setrecursionlimit`; ở bài này `n <= 1000` nên đệ quy ổn.
- Tự dựng test: viết `build_tree(list)` và `tree_to_list(root)` cho mọi bài cây — rất đáng thời gian.

## 15. Biến thể của bài

1. **Path Sum (112) / Path Sum II (113):** đường đi từ gốc tới **lá**; có/liệt kê đường.
2. **Binary Tree Maximum Path Sum (124):** đường đi bất kỳ (có thể đi lên rồi xuống), tổng lớn nhất.
3. **Subarray Sum Equals K (560):** cùng ý tưởng trên mảng.
4. **Đếm đường đi có độ dài đúng `L`** hoặc có tổng `target` **và** kết thúc ở lá.
5. **Trả về chính các đường đi** thay vì số lượng.
6. **Longest ZigZag / Longest Univalue Path (687)**.

In [31]:
def count_paths_ending_at_leaf(root, target):
    """Biến thể 4: chỉ đếm đường đi xuống kết thúc ở LÁ có tổng target."""
    cnt = Counter({0: 1})

    def dfs(node, s):
        if node is None:
            return 0
        s += node.val
        ans = 0
        if node.left is None and node.right is None:
            ans += cnt[s - target]
        cnt[s] += 1
        ans += dfs(node.left, s) + dfs(node.right, s)
        cnt[s] -= 1
        return ans

    return dfs(root, 0)


def list_paths(root, target):
    """Biến thể 5: trả về chính các đường đi (list các giá trị) có tổng target."""
    return [p for p in all_downward_paths(root) if sum(p) == target]


t = build_tree(ex1)
print(count_paths_ending_at_leaf(t, 8))           # 2 : 5->2->1 và -3->11 kết thúc ở lá; 5->3 kết thúc ở nút 3 (không phải lá)
print(list_paths(t, 8))
assert count_paths_ending_at_leaf(t, 8) == 2
assert sorted(list_paths(t, 8)) == sorted([[5, 3], [5, 2, 1], [-3, 11]])

2
[[5, 3], [5, 2, 1], [-3, 11]]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mỗi nút làm điểm bắt đầu | hai tầng đệ quy | `O(n · h)` | `O(h)` | dễ nghĩ, `n` nhỏ |
| 2. Giữ đường đi + cộng ngược | nhìn lên tổ tiên bằng list | `O(n · h)` | `O(h)` | minh hoạ cơ chế tổng tiền tố |
| 3. Tổng tiền tố + hash | `cnt[S − target]` + hoàn tác | `O(n)` | `O(n)` | **nên nộp** |
| 4. Gộp bảng từ dưới lên | mỗi cây con báo cáo mọi tổng xuống | `O(n · h)` | `O(n)` | khi cần thống kê mọi tổng đường đi |
| 5. Tổng tiền tố, lặp | ngăn xếp (vào/ra) | `O(n)` | `O(n)` | cây rất sâu, tránh `RecursionError` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `tree_to_list(build_tree([1, None, 2]))`, `Counter({0: 1})[7]`, `Solution().pathSum(build_tree([3]), 3)`, `Solution().pathSum(build_tree([1, 2]), 3)`, `Solution().pathSum(build_tree([2, 1, 1]), 3)`.

**Bài 2 (tính tay).** Với cây `[1, -2, -3, 1, 3, -2, None, -1]` và `targetSum = -1`: liệt kê mọi đường đi xuống có tổng `-1` (chỉ ra từng đường).

**Bài 3 (điền TODO).** Hoàn thành `path_sum_todo(root, target)` theo Cách 3.

In [32]:
def path_sum_todo(root, target):
    # TODO 1: cnt = Counter({0: 1})
    # TODO 2: viết dfs(node, s): node None -> 0
    #         s += node.val; ans = cnt[s - target]; cnt[s] += 1
    #         ans += dfs(left, s) + dfs(right, s); cnt[s] -= 1; return ans
    # TODO 3: return dfs(root, 0)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `path_sum_iter2(root, target)` theo Cách 5 **không dùng trạng thái `0/1`**, thay vào đó đẩy một dấu hiệu đặc biệt (ví dụ `("undo", s)`) lên ngăn xếp để hoàn tác.

**Bài 5 (biến thể / mini challenge).** (a) Viết `max_path_len_with_sum(root, target)` trả về **độ dài (số nút) lớn nhất** của một đường đi xuống có tổng `target` (nếu không có trả `0`). Gợi ý: lưu trong bảng **độ sâu nhỏ nhất** mà mỗi tổng tiền tố xuất hiện. (b) Viết `count_all_paths(root)` trả về tổng số đường đi xuống (không phụ thuộc tổng); kiểm tra với `len(all_downward_paths(root))`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, None, 2]`, `0`, `1`, `1` (chỉ `1 -> 2`, nút `2` đơn lẻ không bằng 3), `2` (`2 -> 1` bên trái và `2 -> 1` bên phải). Bài 3: bám sát mục 9.
- **Mức 2 (vừa):** Bài 2: có `4` đường đi: `1 -> -2`, `-2 -> 1` (bên trái), `1 -> -2 -> 1 -> -1` và `-1` (nút lá trái nhất). Hãy tự lập bảng như mục 4 rồi đối chiếu bằng `list_paths`. Bài 4: `stack.append(("undo", s))` trước khi đẩy hai con; khi lấy ra `("undo", s)` thì `cnt[s] -= 1`.
- **Mức 3 (sâu):** Bài 5(a): dùng `first_depth = {0: 0}` (tổng tiền tố → độ sâu nhỏ nhất, gốc ảo ở độ sâu `0`); tại nút ở độ sâu `d` với tổng `s`: nếu `s − target` có trong `first_depth` thì độ dài là `d − first_depth[s − target]`; chỉ ghi `first_depth[s]` nếu chưa có, và **xoá khi quay lên nếu chính nút này vừa ghi**. 5(b): ở mỗi nút độ sâu `d` (gốc `d = 1`) có đúng `d` đường đi kết thúc tại nó nên đáp án là `Σ d`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert tree_to_list(build_tree([1, None, 2])) == [1, None, 2]
assert Counter({0: 1})[7] == 0
assert Solution().pathSum(build_tree([3]), 3) == 1
assert Solution().pathSum(build_tree([1, 2]), 3) == 1
assert Solution().pathSum(build_tree([2, 1, 1]), 3) == 2

# Bài 2: liệt kê rồi kiểm tra bằng oracle
tree_b2 = build_tree([1, -2, -3, 1, 3, -2, None, -1])
found = sorted(list_paths(tree_b2, -1))
assert found == sorted([[1, -2], [-2, 1], [1, -2, 1, -1], [-1]])
assert Solution().pathSum(tree_b2, -1) == len(found) == 4
print("Bài 2: các đường đi tổng -1:", found)


# Bài 3
def path_sum_todo_ans(root, target):
    cnt = Counter({0: 1})

    def dfs(node, s):
        if node is None:
            return 0
        s += node.val
        ans = cnt[s - target]
        cnt[s] += 1
        ans += dfs(node.left, s) + dfs(node.right, s)
        cnt[s] -= 1
        return ans

    return dfs(root, 0)


for vals, target, expected in examples:
    assert path_sum_todo_ans(build_tree(vals), target) == expected


# Bài 4
def path_sum_iter2(root, target):
    if root is None:
        return 0
    cnt = Counter({0: 1})
    total = 0
    stack = [(root, 0)]
    while stack:
        item = stack.pop()
        if item[0] == "undo":
            cnt[item[1]] -= 1
            continue
        node, s_parent = item
        s = s_parent + node.val
        total += cnt[s - target]
        cnt[s] += 1
        stack.append(("undo", s))
        if node.right:
            stack.append((node.right, s))
        if node.left:
            stack.append((node.left, s))
    return total


for vals, target, expected in examples:
    assert path_sum_iter2(build_tree(vals), target) == expected


# Bài 5
def max_path_len_with_sum(root, target):
    first_depth = {0: 0}
    best = 0

    def dfs(node, s, d):
        nonlocal best
        if node is None:
            return
        s += node.val
        d += 1
        if s - target in first_depth:
            best = max(best, d - first_depth[s - target])
        added = s not in first_depth
        if added:
            first_depth[s] = d
        dfs(node.left, s, d)
        dfs(node.right, s, d)
        if added:
            del first_depth[s]

    dfs(root, 0, 0)
    return best


def count_all_paths(root):
    def go(node, d):
        return 0 if node is None else d + go(node.left, d + 1) + go(node.right, d + 1)
    return go(root, 1)


def longest_oracle(root, target):
    return max([len(p) for p in all_downward_paths(root) if sum(p) == target], default=0)


assert max_path_len_with_sum(build_tree(ex1), 8) == 3
for _ in range(300):
    r = random_tree(rng.randint(0, 12), -3, 3, chain_bias=rng.choice([0, 0.8]))
    tg = rng.randint(-5, 5)
    assert max_path_len_with_sum(r, tg) == longest_oracle(r, tg)
    assert count_all_paths(r) == len(all_downward_paths(r))
print("Tất cả đáp án bài tập đều đúng.")

Bài 2: các đường đi tổng -1: [[-2, 1], [-1], [1, -2], [1, -2, 1, -1]]
Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm đường đi xuống có tổng `k` bằng tổng tiền tố + bảng băm)

```python
class Solution:
    def pathSum(self, root: Optional[TreeNode], targetSum: int) -> int:
        def dfs(node, s):
            if node is None:
                return 0
            s += node.val
            ans = cnt[s - targetSum]       # tra TRƯỚC
            cnt[s] += 1                    # rồi mới thêm
            ans += dfs(node.left, s)
            ans += dfs(node.right, s)
            cnt[s] -= 1                    # hoàn tác khi quay lên
            return ans

        cnt = Counter({0: 1})              # tổng tiền tố trước gốc
        return dfs(root, 0)
```

**Checklist**

- [ ] Đường đi **không** cần bắt đầu ở gốc / kết thúc ở lá, nhưng phải **đi xuống**?
- [ ] `cnt = Counter({0: 1})` để đường đi bắt đầu ngay ở gốc được tính?
- [ ] Tra `cnt[s - targetSum]` **trước** khi `cnt[s] += 1`?
- [ ] Có `cnt[s] -= 1` sau khi xử lý hai con (quay lui)?
- [ ] Xử lý `root is None` (trả `0`) và giá trị âm/`0` (không cắt tỉa theo tổng)?
- [ ] Đã thử cây rỗng, `[0,0,0]` với target `0` (`5`), ví dụ 1 (`3`) và ví dụ 2 (`3`)?
- [ ] Skeleton đúng `class Solution` / `def pathSum(self, root: Optional[TreeNode], targetSum: int) -> int`?